# 04｜第 1 節：OpenCV＋Tesseract 文字與車牌辨識

**學習方式：依序執行所有 Code Cell，只需修改大寫參數，不必寫新程式。**

這份教材使用 `04_素材/plate_01.png`～`plate_04.png` 四張**已裁切**的車牌照片。OpenCV 做灰階／二值化等處理；Tesseract 負責讀文字。**Tesseract 不會自動從整輛車找到車牌**；上傳整車照片時，需要先調整裁切範圍，或另外使用車牌位置偵測模型。

在 Google Colab 或已連結 Colab 的 VS Code Notebook 執行。請將本 Notebook 與 `04_素材` 放在同一資料夾。第 4 格可以上傳自己的照片，或簡單、清楚的文字圖（含自行製作的驗證碼練習圖）。

## 1. 安裝工具（直接執行）

In [ ]:
# Install the Tesseract OCR application in Colab.
# 在 Colab 安裝 Tesseract OCR 程式。
!apt-get -qq update && apt-get -qq install -y tesseract-ocr

# Install the Python interface for Tesseract.
# 安裝 Python 使用 Tesseract 的套件。
%pip install -q pytesseract

# Import OpenCV for image preprocessing.
# 匯入 OpenCV 處理影像。
import cv2

# Import the OCR interface.
# 匯入文字辨識套件。
import pytesseract

# Import tools to open and show images.
# 匯入開啟與顯示圖片的工具。
from PIL import Image

# Import the Notebook display helper.
# 匯入 Notebook 的圖片顯示工具。
from IPython.display import display

# Import the upload interface for Colab.
# 匯入 Colab 的檔案上傳功能。
from google.colab import files

# Import simple file-path utilities.
# 匯入操作檔案路徑的工具。
from pathlib import Path


## 2. 先看教材車牌圖

只要改 `IMAGE_PATH`，即可切換 `plate_01.png`～`plate_04.png`。照片已經裁切好，所以這裡**沒有進行自動車牌定位**。

In [ ]:
# Choose one of the supplied license-plate images.
# 選擇教材中的一張車牌照片。
IMAGE_PATH = '04_素材/plate_04.png'

# Open the chosen photograph.
# 開啟指定的照片。
image = Image.open(IMAGE_PATH).convert('RGB')

# Show the input photograph.
# 顯示原始照片。
display(image)

# Check its width and height.
# 查看照片的寬度與高度。
print('圖片尺寸：', image.size)


## 3. 調整影像處理與 OCR 參數

修改 `PROCESS_MODE`：`'gray'` 表示灰階，`'binary'` 表示二值化。修改 `THRESHOLD`（0～255）比較辨識差異。`PSM=7` 代表預期是一行文字；對單一短字串也可以試 `PSM=8`。**輸出是真正執行 Tesseract 得到的結果，不保證每張都辨識正確。**

**PSM 的用途對照：**`7` 將文字視為單行（例如裁切後的車牌）；`8` 視為一個單字或短字串；`6` 適合排版整齊的文字區塊；`3` 會自動分析整頁排版。`0` 只進行文字方向與文字系統偵測（OSD），**不會讀出文字內容**，所以不能把 `PSM=0` 當成車牌 OCR 模式。


In [ ]:
# Select grayscale or black-and-white preprocessing.
# 選擇灰階或黑白二值化處理。
PROCESS_MODE = 'gray'

# Set the threshold used only in binary mode.
# 設定黑白模式使用的亮度門檻。
THRESHOLD = 145

# Treat a license plate as one line of text.
# 將車牌視為單行文字進行辨識。
PSM = 7

# Read the selected photo in grayscale.
# 以灰階讀取選取的照片。
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

# Keep grayscale or apply a binary threshold.
# 保留灰階，或轉成黑白二值影像。
processed = gray if PROCESS_MODE == 'gray' else cv2.threshold(gray, THRESHOLD, 255, cv2.THRESH_BINARY)[1]

# Show the image actually sent to OCR.
# 顯示真正送進 OCR 的影像。
display(Image.fromarray(processed))

# Read uppercase letters, numbers, and hyphens.
# 辨識英文字母、數字與連字號。
ocr_text = pytesseract.image_to_string(processed, config=f'--psm {PSM} -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-')

# Print the model's actual OCR output.
# 印出 Tesseract 實際辨識的文字。
print('OCR 結果：', ocr_text.strip())


## 4. 上傳自己的照片，調整裁切範圍

可以上傳**已裁切的車牌**、自己拍攝的車輛照片，或自行製作的簡單文字／驗證碼圖。`CROP` 是 `(左, 上, 右, 下)`，以 **0～1 的比例**指定裁切區域；`(0, 0, 1, 1)` 代表整張圖片。上傳整輛車的照片後，先看顯示的原圖，再修改 `CROP`，重新執行本格，對準車牌。`CUSTOM_PSM`：已裁切的單行車牌可試 `7`，簡單單字或自製的文字圖可試 `8`；如需辨識整齊的多行文字，可嘗試 `6`。`0` 只做方向偵測，不適合讀取字串。

本例只是對自有圖片做 OCR 練習，不代表能可靠破解真實網站的驗證碼。上傳照片涉及真實車牌時，請注意隱私與授權。

In [ ]:
# Upload one photograph from your computer.
# 從自己的電腦上傳一張圖片。
uploaded = files.upload()

# Get the uploaded image filename.
# 取得剛才上傳的圖片檔名。
CUSTOM_IMAGE = next(iter(uploaded))

# Set the crop coordinates as image proportions.
# 使用圖片比例設定裁切位置：左、上、右、下。
CROP = (0.0, 0.0, 1.0, 1.0)

# Choose a line or a single word for OCR.
# 選擇單行文字或單字辨識模式。
CUSTOM_PSM = 7

# Open the uploaded photograph.
# 開啟學生自己上傳的照片。
photo = Image.open(CUSTOM_IMAGE).convert('RGB')

# Show the original image before cropping.
# 先顯示完整原圖，方便調整裁切範圍。
display(photo)

# Read image dimensions and crop proportions.
# 取得圖片尺寸與裁切比例。
width, height = photo.size

# Read the selected crop boundaries.
# 讀取選取的裁切邊界。
left, top, right, bottom = CROP

# Crop the selected region to help OCR focus.
# 裁出需要辨識的文字區域，減少背景干擾。
region = photo.crop((int(width * left), int(height * top), int(width * right), int(height * bottom)))

# Show the cropped image.
# 顯示真正要辨識的裁切區域。
display(region)

# Run OCR on the cropped image.
# 針對裁切區域執行 Tesseract 文字辨識。
custom_text = pytesseract.image_to_string(region, config=f'--psm {CUSTOM_PSM}')

# Print the recognized text.
# 印出上傳圖片的辨識結果。
print('辨識文字：', custom_text.strip())


## 5. 選做：一次比較 4 張車牌

直接執行即可。可回到第 3 格改 `PSM` 或前處理方式，再比較哪一張容易辨識失敗。

In [ ]:
# Find all supplied plate photographs.
# 找出教材中的四張車牌照片。
plate_files = sorted(Path('04_素材').glob('plate_*.png'))

# Read the images one at a time.
# 逐張讀取並辨識車牌照片。
for plate_path in plate_files:
    # Recognize one line from each image.
    # 將每張照片當作一行文字辨識。
    recognized = pytesseract.image_to_string(Image.open(plate_path), config='--psm 7')

    # Compare the filename and OCR output.
    # 比較檔名與文字辨識結果。
    print(plate_path.name, '→', recognized.strip())


**本節重點：**辨識車牌包含「定位車牌區域」和「讀出文字」兩件不同的事；OpenCV 能協助前處理，Tesseract 處理已選取的文字區域。對陰影、反光、傾斜、低解析度或特殊字型，OCR 可能失敗。